# H-09: Temporal Stability & Drift Detection

**Question:** Is the agent's performance stable over time, or is there systematic drift?

| | |
|---|---|
| **H₀ (Null):** | Performance is stable over time — no systematic trend. |
| **Hₐ (Alternative):** | Performance is drifting — systematic trend or structural break exists. |
| **Methods:** | CUSUM (Cumulative Sum) + EWMA (Exponentially Weighted Moving Average) control charts |
| **Scope:** | Per sub-fault, rolled up to category |
| **Data:** | DETECTED-ONLY runs in run order (time-sequential) |

### Key Properties
- **Always active** — no SLA thresholds needed
- **Target** defaults to IQM (inter-quartile mean) per sub-fault
- **EWMA smoothing** λ = 0.2 (default)

### Per-Sub-Fault Verdict Logic
| Condition | Verdict |
|-----------|--------|
| No CUSUM or EWMA alarm | ✅ STABLE |
| CUSUM or EWMA alarm triggered | ❌ DRIFT_DETECTED |
| n < 8 observations | ⚠️ LOW_POWER |

### Category Rollup
| Condition | Category Verdict |
|-----------|------------------|
| All sub-faults STABLE (or mix with LOW_POWER) | STABLE |
| Any sub-fault DRIFT_DETECTED | DRIFT_DETECTED |
| All sub-faults LOW_POWER | LOW_POWER |


In [1]:
import sys, os, json
import numpy as np
from pathlib import Path
from collections import defaultdict

project_root = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(project_root))

data_root = project_root / "hypothesis_framework" / "data" / "input"

def load_runs(category_dir):
    runs = []
    for f in sorted(category_dir.glob("*.json")):
        run = json.loads(f.read_text(encoding="utf-8"))
        runs.append(run)
    return runs

def build_subfault_data(runs, metric_name, detected_only=True):
    grouped = defaultdict(list)
    for run in runs:
        q = run["quantitative"]
        if detected_only and q.get("fault_detected") != "Yes":
            continue
        fname = run["fault_name"]
        val = q.get(metric_name)
        if val is not None:
            grouped[fname].append(float(val))
    return dict(grouped)

# Load runs per category
categories = {}
for cat_name in ["application_fault", "network_fault", "resource_fault"]:
    cat_dir = data_root / cat_name
    if cat_dir.exists():
        categories[cat_name] = load_runs(cat_dir)

print("=== Data Loaded (DETECTED-ONLY, run order) ===")
for cat, runs in categories.items():
    total = len(runs)
    detected = sum(1 for r in runs if r["quantitative"].get("fault_detected") == "Yes")
    subfaults = sorted(set(r["fault_name"] for r in runs))
    print(f"  {cat}: {total} runs, {detected} detected ({detected/total:.0%})")
    for fn in subfaults:
        fn_det = sum(
            1 for r in runs
            if r["fault_name"] == fn
            and r["quantitative"].get("fault_detected") == "Yes"
        )
        print(f"    {fn}: {fn_det} detected")

=== Data Loaded (DETECTED-ONLY, run order) ===
  application_fault: 60 runs, 51 detected (85%)
    container-kill: 28 detected
    pod-delete: 23 detected
  network_fault: 90 runs, 39 detected (43%)
    pod-dns-error: 13 detected
    pod-network-corruption: 14 detected
    pod-network-loss: 12 detected
  resource_fault: 90 runs, 70 detected (78%)
    disk-fill: 25 detected
    pod-cpu-hog: 25 detected
    pod-memory-hog: 20 detected


---
## Step 1: Run H-09 for `time_to_detect`

CUSUM and EWMA control charts per sub-fault on time-ordered `time_to_detect` values.
Target defaults to IQM (inter-quartile mean) of each sub-fault's observations.


In [2]:
from hypothesis_framework.scripts.hypothesis_tests.h09_temporal_stability import run_drift_test

# Build data: {category: {sub_fault: [values in run order]}}
ttd_data = {}
for cat, runs in categories.items():
    ttd_data[cat] = build_subfault_data(runs, "time_to_detect", detected_only=True)

ttd_result = run_drift_test(ttd_data, metric_name="time_to_detect")

# Display per-sub-fault results
verdict_icons = {
    "STABLE": "\u2705 STABLE",
    "DRIFT_DETECTED": "\u274c DRIFT_DETECTED",
    "LOW_POWER": "\u26a0\ufe0f LOW_POWER",
}

print("H-09  time_to_detect — Per-Sub-Fault Drift Results")
print("=" * 90)
print(f"{'Sub-fault':<28} {'n':>4}  {'CUSUM final':>12} {'CUSUM alarm':>12} {'EWMA final':>12} {'EWMA alarm':>11}  Verdict")
print("-" * 90)

for cat_r in ttd_result.per_category:
    cat_icon = verdict_icons.get(cat_r.drift_verdict, cat_r.drift_verdict)
    print(f"\n  [{cat_r.category}]  n={cat_r.n}  sub-faults={cat_r.n_sub_faults}  → {cat_icon}")
    for sf in cat_r.sub_faults:
        icon = verdict_icons.get(sf.drift_verdict, sf.drift_verdict)
        cusum_val = f"{sf.cusum_final:.4f}" if sf.drift_verdict != "LOW_POWER" else "\u2014"
        cusum_alm = str(sf.cusum_alarm) if sf.drift_verdict != "LOW_POWER" else "\u2014"
        ewma_val = f"{sf.ewma_final:.4f}" if sf.drift_verdict != "LOW_POWER" else "\u2014"
        ewma_alm = str(sf.ewma_alarm) if sf.drift_verdict != "LOW_POWER" else "\u2014"
        print(f"    {sf.fault_name:<26} {sf.n:>4}  {cusum_val:>12} {cusum_alm:>12} {ewma_val:>12} {ewma_alm:>11}  {icon}")

print(f"\nOverall: {ttd_result.overall_assessment}")
if ttd_result.warnings:
    print(f"\nWarnings ({len(ttd_result.warnings)}):")
    for w in ttd_result.warnings:
        print(f"  \u26a0 {w}")

H-09  time_to_detect — Per-Sub-Fault Drift Results
Sub-fault                       n   CUSUM final  CUSUM alarm   EWMA final  EWMA alarm  Verdict
------------------------------------------------------------------------------------------

  [application_fault]  n=51  sub-faults=2  → ✅ STABLE
    container-kill               28       24.2297        False     128.3225       False  ✅ STABLE
    pod-delete                   23        0.0000        False     114.3735       False  ✅ STABLE

  [network_fault]  n=39  sub-faults=3  → ✅ STABLE
    pod-dns-error                13      486.9312        False     229.4909       False  ✅ STABLE
    pod-network-corruption       14      223.2052        False     322.6529       False  ✅ STABLE
    pod-network-loss             12      126.3993        False     129.7913       False  ✅ STABLE

  [resource_fault]  n=70  sub-faults=3  → ✅ STABLE
    disk-fill                    25        0.0000        False     215.4501       False  ✅ STABLE
    pod-cpu-hog  

---
## Step 2: Run H-09 for `time_to_mitigate`

Same CUSUM + EWMA analysis on `time_to_mitigate` values.


In [3]:
ttm_data = {}
for cat, runs in categories.items():
    ttm_data[cat] = build_subfault_data(runs, "time_to_mitigate", detected_only=True)

ttm_result = run_drift_test(ttm_data, metric_name="time_to_mitigate")

print("H-09  time_to_mitigate — Per-Sub-Fault Drift Results")
print("=" * 90)
print(f"{'Sub-fault':<28} {'n':>4}  {'CUSUM final':>12} {'CUSUM alarm':>12} {'EWMA final':>12} {'EWMA alarm':>11}  Verdict")
print("-" * 90)

for cat_r in ttm_result.per_category:
    cat_icon = verdict_icons.get(cat_r.drift_verdict, cat_r.drift_verdict)
    print(f"\n  [{cat_r.category}]  n={cat_r.n}  sub-faults={cat_r.n_sub_faults}  → {cat_icon}")
    for sf in cat_r.sub_faults:
        icon = verdict_icons.get(sf.drift_verdict, sf.drift_verdict)
        cusum_val = f"{sf.cusum_final:.4f}" if sf.drift_verdict != "LOW_POWER" else "\u2014"
        cusum_alm = str(sf.cusum_alarm) if sf.drift_verdict != "LOW_POWER" else "\u2014"
        ewma_val = f"{sf.ewma_final:.4f}" if sf.drift_verdict != "LOW_POWER" else "\u2014"
        ewma_alm = str(sf.ewma_alarm) if sf.drift_verdict != "LOW_POWER" else "\u2014"
        print(f"    {sf.fault_name:<26} {sf.n:>4}  {cusum_val:>12} {cusum_alm:>12} {ewma_val:>12} {ewma_alm:>11}  {icon}")

print(f"\nOverall: {ttm_result.overall_assessment}")
if ttm_result.warnings:
    print(f"\nWarnings ({len(ttm_result.warnings)}):")
    for w in ttm_result.warnings:
        print(f"  \u26a0 {w}")

H-09  time_to_mitigate — Per-Sub-Fault Drift Results
Sub-fault                       n   CUSUM final  CUSUM alarm   EWMA final  EWMA alarm  Verdict
------------------------------------------------------------------------------------------

  [application_fault]  n=51  sub-faults=2  → ✅ STABLE
    container-kill               28        0.0000        False     289.0276       False  ✅ STABLE
    pod-delete                   23        0.0000        False     298.7041       False  ✅ STABLE

  [network_fault]  n=39  sub-faults=3  → ✅ STABLE
    pod-dns-error                13      440.7305        False     481.1490       False  ✅ STABLE
    pod-network-corruption       14        0.0000        False     569.9713       False  ✅ STABLE
    pod-network-loss             12      145.7003        False     494.5198       False  ✅ STABLE

  [resource_fault]  n=70  sub-faults=3  → ✅ STABLE
    disk-fill                    25       60.1141        False     459.6691       False  ✅ STABLE
    pod-cpu-hog

---
## Step 3: Summary & Interpretation

Combines drift results from both metrics into a unified view.


In [4]:
print("H-09  Temporal Stability — Summary")
print("=" * 70)

for label, result in [("time_to_detect", ttd_result), ("time_to_mitigate", ttm_result)]:
    print(f"\n  Metric: {label}")
    print(f"  Overall: {result.overall_assessment}")
    for cat_r in result.per_category:
        icon = verdict_icons.get(cat_r.drift_verdict, cat_r.drift_verdict)
        stable_n = sum(1 for s in cat_r.sub_faults if s.drift_verdict == "STABLE")
        drift_n = sum(1 for s in cat_r.sub_faults if s.drift_verdict == "DRIFT_DETECTED")
        low_n = sum(1 for s in cat_r.sub_faults if s.drift_verdict == "LOW_POWER")
        print(f"    {cat_r.category:<24} {icon:<22}  (stable={stable_n}, drift={drift_n}, low_power={low_n})")

print("\n" + "-" * 70)
print("Interpretation:")
print("  ✅ STABLE       — No systematic trend detected; performance is consistent.")
print("  ❌ DRIFT_DETECTED — CUSUM or EWMA alarm fired; investigate for degradation.")
print("  ⚠️ LOW_POWER    — Fewer than 8 runs; insufficient data for drift detection.")

H-09  Temporal Stability — Summary

  Metric: time_to_detect
  Overall: no_drift_detected
    application_fault        ✅ STABLE                (stable=2, drift=0, low_power=0)
    network_fault            ✅ STABLE                (stable=3, drift=0, low_power=0)
    resource_fault           ✅ STABLE                (stable=3, drift=0, low_power=0)

  Metric: time_to_mitigate
  Overall: no_drift_detected
    application_fault        ✅ STABLE                (stable=2, drift=0, low_power=0)
    network_fault            ✅ STABLE                (stable=3, drift=0, low_power=0)
    resource_fault           ✅ STABLE                (stable=3, drift=0, low_power=0)

----------------------------------------------------------------------
Interpretation:
  ✅ STABLE       — No systematic trend detected; performance is consistent.
  ❌ DRIFT_DETECTED — CUSUM or EWMA alarm fired; investigate for degradation.
  ⚠️ LOW_POWER    — Fewer than 8 runs; insufficient data for drift detection.


---
## Step 4: JSON Output

Serialized H09Result objects for downstream pipeline consumption.


In [5]:
print("=== time_to_detect ===")
print(json.dumps(ttd_result.model_dump(), indent=2, default=str, ensure_ascii=False))

print("\n=== time_to_mitigate ===")
print(json.dumps(ttm_result.model_dump(), indent=2, default=str, ensure_ascii=False))

=== time_to_detect ===
{
  "hypothesis_id": "H-09",
  "hypothesis_name": "Temporal Stability & Drift Detection",
  "metric_name": "time_to_detect",
  "null_hypothesis": "Performance is stable over time — no systematic trend.",
  "alt_hypothesis": "Performance is drifting — systematic trend or structural break exists.",
  "alpha": 0.05,
  "overall_assessment": "no_drift_detected",
  "warnings": [
    "n=28 < 30; CUSUM/EWMA have limited power. Interpret drift verdicts as directional indicators.",
    "n=23 < 30; CUSUM/EWMA have limited power. Interpret drift verdicts as directional indicators.",
    "n=13 < 30; CUSUM/EWMA have limited power. Interpret drift verdicts as directional indicators.",
    "n=14 < 30; CUSUM/EWMA have limited power. Interpret drift verdicts as directional indicators.",
    "n=12 < 30; CUSUM/EWMA have limited power. Interpret drift verdicts as directional indicators.",
    "n=25 < 30; CUSUM/EWMA have limited power. Interpret drift verdicts as directional indicator